# 03 — Platform Analysis

Analyse du support des plateformes (Windows, Mac, Linux) sur le marché Steam.

**Objectif métier** : Comprendre les priorités de développement multi-plateforme et identifier les opportunités sur Mac/Linux.

In [0]:
from pyspark.sql import functions as F

df = spark.table("steam_clean")
total_games = df.count()
print(f"Total games: {total_games}")

Total games: 55690


## Q11 — What is the availability of games on Windows, Mac and Linux?

In [0]:
df_platforms = df.select(
    F.sum(F.col("platform_windows").cast("int")).alias("Windows"),
    F.sum(F.col("platform_mac").cast("int")).alias("Mac"),
    F.sum(F.col("platform_linux").cast("int")).alias("Linux"),
    F.count("*").alias("Total")
)
display(df_platforms)

Windows,Mac,Linux,Total
55675,12769,8457,55690


In [0]:
df_pct = df.select(
    F.round(F.sum(F.col("platform_windows").cast("int")) / F.lit(total_games) * 100, 1).alias("Windows_pct"),
    F.round(F.sum(F.col("platform_mac").cast("int")) / F.lit(total_games) * 100, 1).alias("Mac_pct"),
    F.round(F.sum(F.col("platform_linux").cast("int")) / F.lit(total_games) * 100, 1).alias("Linux_pct"),
)
display(df_pct)

row = df_pct.first()
overall_mac_pct = row["Mac_pct"]
overall_linux_pct = row["Linux_pct"]

pct_win = F.round(F.sum(F.col("platform_windows").cast("int")) / F.count("*") * 100, 1).alias("pct_windows")
pct_mac = F.round(F.sum(F.col("platform_mac").cast("int")) / F.count("*") * 100, 1).alias("pct_mac")
pct_lnx = F.round(F.sum(F.col("platform_linux").cast("int")) / F.count("*") * 100, 1).alias("pct_linux")

Windows_pct,Mac_pct,Linux_pct
100.0,22.9,15.2


In [0]:
df_combos = (df
    .withColumn("combo",
        F.concat_ws("-",
            F.when(F.col("platform_windows"), F.lit("Win")),
            F.when(F.col("platform_mac"), F.lit("Mac")),
            F.when(F.col("platform_linux"), F.lit("Linux"))
        )
    )
    .groupBy("combo")
    .agg(F.count("*").alias("count"))
    .withColumn("pct", F.round(F.col("count") / F.lit(total_games) * 100, 1))
    .orderBy(F.desc("count"))
)
display(df_combos)

combo,count,pct
Win,41271,74.1
Win-Mac-Linux,6806,12.2
Win-Mac,5951,10.7
Win-Linux,1647,3.0
Mac,11,0.0
Linux,3,0.0
Mac-Linux,1,0.0


Databricks visualization. Run in Databricks to view.

In [0]:
df_combo_payoff = (df
    .withColumn("combo", F.concat_ws("-",
        F.when(F.col("platform_windows"), F.lit("Win")),
        F.when(F.col("platform_mac"), F.lit("Mac")),
        F.when(F.col("platform_linux"), F.lit("Linux"))
    ))
    .groupBy("combo")
    .agg(
        F.count("*").alias("game_count"),
        F.round(F.avg("avg_owners"), 0).alias("avg_owners"),
        F.round(F.percentile_approx("avg_owners", 0.5), 0).alias("median_owners"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.avg("estimated_revenue"), 0).alias("avg_estimated_revenue")
    )
    .orderBy(F.desc("avg_estimated_revenue"))
)
display(df_combo_payoff)

combo,game_count,avg_owners,median_owners,avg_rating,avg_estimated_revenue
Win-Mac-Linux,6806,319782.0,10000,0.783,3116984.0
Win-Mac,5951,139744.0,10000,0.77,1586537.0
Win,41271,102156.0,10000,0.722,1426356.0
Win-Linux,1647,127984.0,10000,0.782,1384526.0
Mac-Linux,1,75000.0,75000,0.841,374250.0
Linux,3,18333.0,10000,0.705,349883.0
Mac,11,27727.0,10000,0.46,270400.0


Databricks visualization. Run in Databricks to view.

## Complement Q11 — Historical evolution of multi-platform support

In [0]:
df_yearly_platform = (df
    .filter(F.col("release_year").isNotNull())
    .filter(F.col("release_year") >= 2010)
    .groupBy("release_year")
    .agg(
        F.count("*").alias("total_games"),
        pct_win, pct_mac, pct_lnx
    )
    .orderBy("release_year")
)
display(df_yearly_platform)

release_year,total_games,pct_windows,pct_mac,pct_linux
2010,288,100.0,34.0,20.5
2011,267,100.0,30.7,18.0
2012,345,99.7,41.7,27.2
2013,471,100.0,46.7,38.2
2014,1557,100.0,42.5,29.7
2015,2575,100.0,41.9,30.6
2016,4185,100.0,34.6,23.9
2017,6017,99.9,27.2,18.3
2018,7678,100.0,24.2,14.6
2019,6968,100.0,20.9,12.3


Databricks visualization. Run in Databricks to view.

## Q12 — Which genres are preferentially available on each platform?

In [0]:
# `genre` has 160 NULLs (0.3% excluded)
df_genres = (df
    .filter(F.col("genre").isNotNull())
    .withColumn("genre_single", F.explode(F.col("genre")))
)

df_genre_platform = (df_genres
    .groupBy("genre_single")
    .agg(
        F.count("*").alias("total"),
        pct_win, pct_mac, pct_lnx
    )
    .filter(F.col("total") >= 500)
    .withColumn("mac_deviation", F.round(F.col("pct_mac") - F.lit(overall_mac_pct), 1))
    .withColumn(
        "linux_deviation",
        F.round(F.col("pct_linux") - F.lit(overall_linux_pct), 1),
    )
    .orderBy(F.desc("pct_linux"))
)
display(df_genre_platform)

genre_single,total,pct_windows,pct_mac,pct_linux,mac_deviation,linux_deviation
Indie,39681,100.0,25.0,17.6,2.1,2.4
Strategy,10895,100.0,27.6,16.8,4.7,1.6
RPG,9534,100.0,23.6,16.0,0.7,0.8
Adventure,21431,100.0,23.5,15.4,0.6,0.2
Casual,22086,100.0,23.2,15.0,0.3,-0.2
Action,23759,100.0,19.2,14.2,-3.7,-1.0
Simulation,10836,100.0,22.5,14.1,-0.4,-1.1
Racing,2155,100.0,19.7,14.1,-3.2,-1.1
Free to Play,3393,99.9,24.9,14.0,2.0,-1.2
Massively Multiplayer,1460,99.9,18.5,11.2,-4.4,-4.0


Databricks visualization. Run in Databricks to view.

In [0]:
df_mac_genres = df_genre_platform.orderBy(F.desc("pct_mac")).limit(10)
display(df_mac_genres)

genre_single,total,pct_windows,pct_mac,pct_linux,mac_deviation,linux_deviation
Strategy,10895,100.0,27.6,16.8,4.7,1.6
Indie,39681,100.0,25.0,17.6,2.1,2.4
Free to Play,3393,99.9,24.9,14.0,2.0,-1.2
RPG,9534,100.0,23.6,16.0,0.7,0.8
Adventure,21431,100.0,23.5,15.4,0.6,0.2
Casual,22086,100.0,23.2,15.0,0.3,-0.2
Simulation,10836,100.0,22.5,14.1,-0.4,-1.1
Racing,2155,100.0,19.7,14.1,-3.2,-1.1
Action,23759,100.0,19.2,14.2,-3.7,-1.0
Sports,2666,100.0,19.0,10.8,-3.9,-4.4


Databricks visualization. Run in Databricks to view.

In [0]:
df_linux_genres = df_genre_platform.orderBy(F.desc("pct_linux")).limit(10)
display(df_linux_genres)

genre_single,total,pct_windows,pct_mac,pct_linux,mac_deviation,linux_deviation
Indie,39681,100.0,25.0,17.6,2.1,2.4
Strategy,10895,100.0,27.6,16.8,4.7,1.6
RPG,9534,100.0,23.6,16.0,0.7,0.8
Adventure,21431,100.0,23.5,15.4,0.6,0.2
Casual,22086,100.0,23.2,15.0,0.3,-0.2
Action,23759,100.0,19.2,14.2,-3.7,-1.0
Racing,2155,100.0,19.7,14.1,-3.2,-1.1
Simulation,10836,100.0,22.5,14.1,-0.4,-1.1
Free to Play,3393,99.9,24.9,14.0,2.0,-1.2
Massively Multiplayer,1460,99.9,18.5,11.2,-4.4,-4.0


Databricks visualization. Run in Databricks to view.

## Complementary — Category analysis (Multiplayer, VR, etc.)

In [0]:
# `categories` has 970 NULLs (1.7% excluded)
df_categories = (df
    .filter(F.col("categories").isNotNull())
    .withColumn("category_single", F.explode(F.col("categories")))
)

df_cat_counts = (df_categories
    .groupBy("category_single")
    .agg(F.count("*").alias("game_count"))
    .orderBy(F.desc("game_count"))
    .limit(15)
)
display(df_cat_counts)

category_single,game_count
Single-player,52025
Steam Achievements,27394
Steam Cloud,14235
Full controller support,11878
Multi-player,11455
Steam Trading Cards,9208
Partial Controller Support,7867
PvP,7070
Co-op,5616
Steam Leaderboards,5509


Databricks visualization. Run in Databricks to view.

In [0]:
df_vr = df_categories.filter(F.col("category_single").contains("VR"))
vr_count = df_vr.select("appid").distinct().count()
print(f"Games with VR support: {vr_count}")

df_vr_platform = (df_vr
    .select("appid", "platform_windows", "platform_mac", "platform_linux")
    .distinct()
    .select(
        F.count("*").alias("vr_games"),
        pct_win, pct_mac, pct_lnx
    )
)
display(df_vr_platform)

Games with VR support: 261


vr_games,pct_windows,pct_mac,pct_linux
261,99.6,23.8,18.4


## Conclusion — Analyse des plateformes (Q11-Q12)

### Q11 — Disponibilite des jeux sur Windows, Mac et Linux

**Couverture globale** (sur 55 690 jeux) :
- Windows : 55 675 jeux, soit 99.97% — quasi-universel, mais pas 100% (15 jeux non-Windows : Mac-only, Linux-only ou Mac-Linux)
- Mac : 12 769 jeux, soit 22.9%
- Linux : 8 457 jeux, soit 15.2%

**Combinaisons de plateformes** :
- Windows-only : 41 271 jeux (74.1%) — configuration dominante
- Win-Mac-Linux : 6 806 jeux (12.2%) — seul vrai multi-plateforme complet
- Win-Mac : 5 951 jeux (10.7%)
- Win-Linux : 1 647 jeux (3.0%)
- Mac-only / Linux-only / Mac-Linux : 15 jeux au total (negligeable)

**Evolution temporelle (2010-2022)** : le support Mac et Linux a atteint un pic en 2013 (Mac 46.7%, Linux 38.2%) suivi d'une tendance a la baisse. En 2022 : Mac 14.0%, Linux 10.6%. L'ecart Mac-Linux s'est reduit de 8.5 points en 2013 a 3.4 points en 2022. Windows reste stable a ~100% sur toute la periode (99.7% a 100.0%).

**Complement — Categories Steam** :
- Single-player domine (52 025 jeux), suivi de Steam Achievements (27 394) et Steam Cloud (14 235)
- Multi-player : 11 455 jeux ; Co-op : 5 616 jeux
- VR : marche niche de 261 jeux, avec un support plateforme proche de la moyenne (Windows 99.6%, Mac 23.8%, Linux 18.4%)

### Q12 — Genres preferentiels par plateforme

Analyse sur 13 genres ayant au moins 500 jeux (160 NULLs exclus, soit 0.3%).

**Genres sur-representes sur Mac** (deviation positive par rapport a la moyenne de 22.9%) :
- Strategy : 27.6% (+4.7 pts) — genre le plus porte sur Mac
- Indie : 25.0% (+2.1 pts)
- Free to Play : 24.9% (+2.0 pts)
- RPG : 23.6% (+0.7 pt), Adventure : 23.5% (+0.6 pt) — proches de la moyenne

**Genres sous-representes sur Mac** :
- Early Access : 14.6% (-8.3 pts) — plus fort deficit
- Utilities : 15.0% (-7.9 pts)
- Massively Multiplayer : 18.5% (-4.4 pts)
- Sports : 19.0% (-3.9 pts), Action : 19.2% (-3.7 pts)

**Genres sur-representes sur Linux** (deviation positive par rapport a la moyenne de 15.2%) :
- Indie : 17.6% (+2.4 pts) — genre le plus porte sur Linux
- Strategy : 16.8% (+1.6 pt)
- RPG : 16.0% (+0.8 pt)

**Genres sous-representes sur Linux** :
- Utilities : 7.2% (-8.0 pts) — plus fort deficit
- Early Access : 10.3% (-4.9 pts)
- Sports : 10.8% (-4.4 pts)
- Massively Multiplayer : 11.2% (-4.0 pts)

**Convergence Mac/Linux** : les memes genres sont favorises (Strategy, Indie, RPG) et defavorises (Early Access, Utilities, Sports, Massively Multiplayer) sur les deux plateformes. Les studios qui portent sur Mac tendent a porter aussi sur Linux.

### Limites

- Couverture genre : 160 NULLs exclus (0.3% du dataset) — impact negligeable
- Couverture categories : 970 NULLs exclus (1.7% du dataset) — impact faible
- Les deviations par genre restent modestes (max +4.7 pts pour Strategy/Mac, max -8.3 pts pour Early Access/Mac), ce qui limite la portee des conclusions sur les preferences de plateforme par genre
- L'arrondi a 1 decimale (F.round(..., 1)) sur les pourcentages masque les ecarts fins : Windows affiche 100.0% alors que la valeur exacte est 99.97%